In [14]:
import pandas as pd

train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")

In [ ]:
train.describe(include='all')

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

num_cols = train.select_dtypes(include=['int64', 'float64']).columns

train[num_cols].hist(figsize=(16, 10), bins=30)
plt.suptitle("Numeric distributions")
plt.show()

In [ ]:
#cat_cols = df.select_dtypes(include=['object', 'category']).columns

#for col in cat_cols:
#    print(f"\n{col}")
#    print(df[col].unique())

In [15]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    train.drop("class", axis=1), train["class"],
    test_size=0.2,
    random_state=42
)

In [19]:
import numpy as np
import pandas as pd

def feature_engineering(df):
    df = df.copy()

    # ======================
    # 0. SAFE NUMERIC CONVERSION
    # ======================
    num_cols = ["u", "g", "r", "i", "z", "redshift", "alpha", "delta"]

    for col in num_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # ======================
    # 1. COLOR INDICES
    # ======================
    df["u_g"] = df["u"] - df["g"]
    df["g_r"] = df["g"] - df["r"]
    df["r_i"] = df["r"] - df["i"]
    df["i_z"] = df["i"] - df["z"]

    df["u_r"] = df["u"] - df["r"]
    df["g_z"] = df["g"] - df["z"]

    # ======================
    # 2. REDSHIFT FEATURES (SAFE)
    # ======================
    df["redshift"] = df["redshift"].clip(lower=0)

    df["redshift_log"] = np.log1p(df["redshift"])
    df["redshift_sq"] = df["redshift"] ** 2

    # ======================
    # 3. CYCLIC ENCODING
    # ======================
    df["alpha_rad"] = np.radians(df["alpha"])
    df["delta_rad"] = np.radians(df["delta"])

    df["alpha_sin"] = np.sin(df["alpha_rad"])
    df["alpha_cos"] = np.cos(df["alpha_rad"])
    df["delta_sin"] = np.sin(df["delta_rad"])
    df["delta_cos"] = np.cos(df["delta_rad"])

    df.drop(columns=["alpha_rad", "delta_rad"], inplace=True)

    # ======================
    # 4. DROP ID
    # ======================
    if "id" in df.columns:
        df = df.drop(columns=["id"])

    # ======================
    # 5. FINAL CLEANING
    # ======================
    df = df.replace([np.inf, -np.inf], np.nan)

    return df

In [20]:
X_train_fe = feature_engineering(X_train)
X_test_fe = feature_engineering(X_test)
test_fe = feature_engineering(test)

In [21]:
num_cols = X_train_fe.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X_train_fe.select_dtypes(include=["object", "category"]).columns

C:\Users\adik\AppData\Local\Temp\ipykernel_19092\3875277299.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_train_fe.select_dtypes(include=["object", "category"]).columns


In [ ]:
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier
from sklearn.preprocessing import PolynomialFeatures


preprocess = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
    ]), num_cols),

    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("ohe", OneHotEncoder(
            handle_unknown="ignore",
            min_frequency=0.01,
            max_categories=100
        ))
    ]), cat_cols)
])

from xgboost import XGBClassifier

clf = XGBClassifier(
    tree_method="hist",
    device="cuda",

    objective="multi:softprob",
    num_class=3,

    n_estimators=3000,
    learning_rate=0.02,

    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,

    reg_alpha=0.1,
    reg_lambda=2.0,

    random_state=42
)

pipeline = Pipeline([
    ("preprocess", preprocess),
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
    ("clf", clf)
])

In [22]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

In [ ]:
pipeline.fit(X_train_fe, y_train_enc)

In [ ]:
import os
import joblib
import numpy as np
from datetime import datetime
from sklearn.metrics import (
    classification_report,
    f1_score,
    accuracy_score,
    precision_score,
    recall_score,
    roc_auc_score,
    log_loss,
    confusion_matrix
)

def full_metrics(model, X_test, y_test, name="model", save_dir="models"):
    os.makedirs(save_dir, exist_ok=True)

    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)

    # ================= METRICS =================
    acc = accuracy_score(y_test, y_pred)
    f1_w = f1_score(y_test, y_pred, average="weighted")
    f1_m = f1_score(y_test, y_pred, average="macro")
    prec = precision_score(y_test, y_pred, average="weighted")
    rec = recall_score(y_test, y_pred, average="weighted")
    roc = roc_auc_score(y_test, y_proba, multi_class="ovr", average="macro")
    ll = log_loss(y_test, y_proba)
    report = classification_report(y_test, y_pred, digits=6)
    cm = confusion_matrix(y_test, y_pred)

    # ================= PRINT =================
    print("\n================ METRICS ================\n")
    print("Accuracy:", acc)
    print("F1 weighted:", f1_w)
    print("F1 macro:", f1_m)
    print("Precision weighted:", prec)
    print("Recall weighted:", rec)
    print("ROC-AUC macro OVR:", roc)
    print("Log Loss:", ll)

    print("\nClassification report:\n")
    print(report)

    print("\nConfusion matrix:\n")
    print(cm)

    # ================= SAVE MODEL =================
    model_path = os.path.join(save_dir, f"{name}.pkl")
    joblib.dump(model, model_path)

    # ================= SAVE TXT REPORT =================
    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    txt_path = os.path.join(save_dir, f"{name}_{timestamp}.txt")

    with open(txt_path, "w", encoding="utf-8") as f:
        f.write("===== METRICS =====\n")
        f.write(f"Accuracy: {acc}\n")
        f.write(f"F1 weighted: {f1_w}\n")
        f.write(f"F1 macro: {f1_m}\n")
        f.write(f"Precision weighted: {prec}\n")
        f.write(f"Recall weighted: {rec}\n")
        f.write(f"ROC-AUC: {roc}\n")
        f.write(f"Log Loss: {ll}\n\n")

        f.write("===== CLASSIFICATION REPORT =====\n")
        f.write(report + "\n")

        f.write("===== CONFUSION MATRIX =====\n")
        f.write(np.array2string(cm))

    print(f"\nSaved model → {model_path}")
    print(f"Saved report → {txt_path}")

In [ ]:
full_metrics(pipeline, X_test_fe, y_test_enc, name="XGB_v5")

In [24]:
# PREDICTION

import joblib

pipeline = joblib.load("models/XGB_v5.pkl")

test_preds_enc = pipeline.predict(test_fe)
test_preds = le.inverse_transform(test_preds_enc)

# SUBMISSION
submission = pd.DataFrame({
    "id": test["id"],
    "class": test_preds
})

submission.to_csv("submission.csv", index=False)